# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [2]:
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity

# # Load pre-trained model
# model = SentenceTransformer('all-MiniLM-L6-v2')

# # Sentence pairs
# sentence_pairs = [
#     ("A dog is playing in the park.", "A dog is running in a field."),
#     ("I love pizza.", "I enjoy ice cream."),
#     ("What is AI?", "How does a computer learn?")]

# Compute similarities

#YOUR CODE HERE
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Load pre-trained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# Flatten sentences for encoding
sentences = [s for pair in sentence_pairs for s in pair]

# Generate embeddings
embeddings = model.encode(sentences)

# Compute similarities
results = []

for i, (s1, s2) in enumerate(sentence_pairs):
    emb1 = embeddings[2*i].reshape(1, -1)
    emb2 = embeddings[2*i + 1].reshape(1, -1)

    similarity = cosine_similarity(emb1, emb2)[0][0]

    results.append({
        "Sentence 1": s1,
        "Sentence 2": s2,
        "Cosine Similarity": similarity
    })

df = pd.DataFrame(results)

print(df)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                      Sentence 1                    Sentence 2  \
0  A dog is playing in the park.  A dog is running in a field.   
1                  I love pizza.            I enjoy ice cream.   
2                    What is AI?    How does a computer learn?   

   Cosine Similarity  
0           0.521975  
1           0.528068  
2           0.319435  


In [3]:
# More interesting and professional - pairs + similarity matrix!

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sentence pairs for Task 1
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# ---- Pairwise Similarity ----
results = []

for s1, s2 in sentence_pairs:
    embeddings = model.encode([s1, s2])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    results.append((s1, s2, similarity))

pairwise_df = pd.DataFrame(
    results,
    columns=["Sentence 1", "Sentence 2", "Cosine Similarity"]
)

print("\nPairwise Semantic Similarity\n")
display(pairwise_df)

# ---- Similarity Matrix ----

sentences = [s for pair in sentence_pairs for s in pair]

embeddings = model.encode(sentences)

similarity_matrix = cosine_similarity(embeddings)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=sentences,
    columns=sentences
)

print("\nFull Similarity Matrix\n")
display(similarity_df)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Pairwise Semantic Similarity



,Sentence 1,Sentence 2,Cosine Similarity
0,A dog is playing in the park.,A dog is running in a field.,0.521975
1,I love pizza.,I enjoy ice cream.,0.528068
2,What is AI?,How does a computer learn?,0.319435



Full Similarity Matrix



,A dog is playing in the park.,A dog is running in a field.,I love pizza.,I enjoy ice cream.,What is AI?,How does a computer learn?
A dog is playing in the park.,1.000000,0.521975,0.060507,0.095402,0.030728,0.048714
A dog is running in a field.,0.521975,1.000001,0.021024,-0.078276,0.092099,0.016920
I love pizza.,0.060507,0.021024,1.000000,0.528068,0.015717,0.044141
I enjoy ice cream.,0.095402,-0.078276,0.528068,1.000000,0.149976,0.005265
What is AI?,0.030728,0.092099,0.015717,0.149976,1.000000,0.319435
How does a computer learn?,0.048714,0.016920,0.044141,0.005265,0.319435,1.000000


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import pandas as pd

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Example documents
documents = [
    "Artificial intelligence is transforming healthcare.",
    "Machine learning models can detect diseases early.",
    "Doctors use AI to analyze medical images.",
    
    "Football is the most popular sport in the world.",
    "The team scored three goals in the match.",
    "The championship game was exciting.",
    
    "Python is widely used in data science.",
    "Pandas helps analyze tabular datasets.",
    "Machine learning libraries simplify model building."
]

# Generate embeddings
embeddings = model.encode(documents)

# Apply KMeans clustering
num_clusters = 3
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

clusters = kmeans.fit_predict(embeddings)

# Store results
df = pd.DataFrame({
    "Document": documents,
    "Cluster": clusters
})

display(df)

# Inspect clusters
for cluster_id in range(num_clusters):
    print(f"\nCluster {cluster_id}")
    print("-" * 40)
    for doc in df[df["Cluster"] == cluster_id]["Document"]:
        print(doc)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Document,Cluster
0,Artificial intelligence is transforming health...,2
1,Machine learning models can detect diseases ea...,2
2,Doctors use AI to analyze medical images.,2
3,Football is the most popular sport in the world.,0
4,The team scored three goals in the match.,0
5,The championship game was exciting.,0
6,Python is widely used in data science.,1
7,Pandas helps analyze tabular datasets.,1
8,Machine learning libraries simplify model buil...,1



Cluster 0
----------------------------------------
Football is the most popular sport in the world.
The team scored three goals in the match.
The championship game was exciting.

Cluster 1
----------------------------------------
Python is widely used in data science.
Pandas helps analyze tabular datasets.
Machine learning libraries simplify model building.

Cluster 2
----------------------------------------
Artificial intelligence is transforming healthcare.
Machine learning models can detect diseases early.
Doctors use AI to analyze medical images.


In [5]:
# from sklearn.cluster import KMeans

# # Documents to cluster
# documents = [
#     "What is the capital of France?",
#     "How do I bake a chocolate cake?",
#     "What is the distance between Earth and Mars?",
#     "How do I change a flat tire on a car?",
#     "What is the best way to learn Python?",
#     "How do I fix a leaky faucet?"]

# Encode documents

#YOUR CODE HERE
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import pandas as pd

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Example documents
documents = [
    "Artificial intelligence is transforming healthcare.",
    "Machine learning models can detect diseases early.",
    "Doctors use AI to analyze medical images.",
    
    "Football is the most popular sport in the world.",
    "The team scored three goals in the match.",
    "The championship game was exciting.",
    
    "Python is widely used in data science.",
    "Pandas helps analyze tabular datasets.",
    "Machine learning libraries simplify model building."
]

# Generate embeddings
embeddings = model.encode(documents)

# Apply KMeans clustering
num_clusters = 3
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

clusters = kmeans.fit_predict(embeddings)

# Store results
df = pd.DataFrame({
    "Document": documents,
    "Cluster": clusters
})

display(df)

# Inspect clusters
for cluster_id in range(num_clusters):
    print(f"\nCluster {cluster_id}")
    print("-" * 40)
    for doc in df[df["Cluster"] == cluster_id]["Document"]:
        print(doc)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Document,Cluster
0,Artificial intelligence is transforming health...,2
1,Machine learning models can detect diseases ea...,2
2,Doctors use AI to analyze medical images.,2
3,Football is the most popular sport in the world.,0
4,The team scored three goals in the match.,0
5,The championship game was exciting.,0
6,Python is widely used in data science.,1
7,Pandas helps analyze tabular datasets.,1
8,Machine learning libraries simplify model buil...,1



Cluster 0
----------------------------------------
Football is the most popular sport in the world.
The team scored three goals in the match.
The championship game was exciting.

Cluster 1
----------------------------------------
Python is widely used in data science.
Pandas helps analyze tabular datasets.
Machine learning libraries simplify model building.

Cluster 2
----------------------------------------
Artificial intelligence is transforming healthcare.
Machine learning models can detect diseases early.
Doctors use AI to analyze medical images.


In [6]:
# Perform KMeans clustering

#YOUR CODE HERE
from sklearn.cluster import KMeans
import pandas as pd

# Number of clusters
num_clusters = 3

# Initialize KMeans
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

# Fit model and assign clusters
clusters = kmeans.fit_predict(embeddings)

# Create results dataframe
df = pd.DataFrame({
    "Document": documents,
    "Cluster": clusters
})

# Sort for readability
df = df.sort_values("Cluster").reset_index(drop=True)

df

,Document,Cluster
0,Football is the most popular sport in the world.,0
1,The team scored three goals in the match.,0
2,The championship game was exciting.,0
3,Python is widely used in data science.,1
4,Pandas helps analyze tabular datasets.,1
5,Machine learning libraries simplify model buil...,1
6,Artificial intelligence is transforming health...,2
7,Doctors use AI to analyze medical images.,2
8,Machine learning models can detect diseases ea...,2


In [7]:
# Print cluster assignments

#YOUR CODE HERE
from sklearn.cluster import KMeans
import pandas as pd

# Number of clusters
num_clusters = 3

# Initialize and fit KMeans
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
clusters = kmeans.fit_predict(embeddings)

# Store results
df = pd.DataFrame({
    "Document": documents,
    "Cluster": clusters
})

# Sort for readability
df = df.sort_values("Cluster").reset_index(drop=True)

print("\nDocument Cluster Assignments\n")
display(df)

# Analyze clusters
print("\nCluster Analysis\n")

for cluster_id in range(num_clusters):
    print(f"Cluster {cluster_id}")
    print("-" * 40)
    
    cluster_docs = df[df["Cluster"] == cluster_id]["Document"]
    
    for doc in cluster_docs:
        print(doc)
        
    print()


Document Cluster Assignments



,Document,Cluster
0,Football is the most popular sport in the world.,0
1,The team scored three goals in the match.,0
2,The championship game was exciting.,0
3,Python is widely used in data science.,1
4,Pandas helps analyze tabular datasets.,1
5,Machine learning libraries simplify model buil...,1
6,Artificial intelligence is transforming health...,2
7,Doctors use AI to analyze medical images.,2
8,Machine learning models can detect diseases ea...,2



Cluster Analysis

Cluster 0
----------------------------------------
Football is the most popular sport in the world.
The team scored three goals in the match.
The championship game was exciting.

Cluster 1
----------------------------------------
Python is widely used in data science.
Pandas helps analyze tabular datasets.
Machine learning libraries simplify model building.

Cluster 2
----------------------------------------
Artificial intelligence is transforming healthcare.
Doctors use AI to analyze medical images.
Machine learning models can detect diseases early.



### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [8]:
# import numpy as np

# # Documents dataset
# documents = [
#     "What is the capital of France?",
#     "How do I bake a chocolate cake?",
#     "What is the distance between Earth and Mars?",
#     "How do I change a flat tire on a car?",
#     "What is the best way to learn Python?",
#     "How do I fix a leaky faucet?",
#     "What are the best travel destinations in Europe?",
#     "How do I set up a local server?",
#     "What is quantum computing?",
#     "How do I build a mobile app?"
# ]

# Compute document embeddings

#YOUR CODE HERE
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Document dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Encode documents once (important for search systems)
doc_embeddings = model.encode(documents)

# User query
query = "How can I start programming in Python?"

# Encode query
query_embedding = model.encode([query])

# Compute similarity scores
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

# Rank results
results = pd.DataFrame({
    "Document": documents,
    "Similarity": similarities
})

results = results.sort_values(by="Similarity", ascending=False).reset_index(drop=True)

# Return top 5 results
top_results = results.head(5)

print("\nQuery:", query)
print("\nTop 5 Semantic Search Results\n")
display(top_results)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: How can I start programming in Python?

Top 5 Semantic Search Results



,Document,Similarity
0,What is the best way to learn Python?,0.770831
1,How do I build a mobile app?,0.255857
2,What is quantum computing?,0.203234
3,How do I set up a local server?,0.195256
4,What is the capital of France?,0.138186


In [9]:
# # Create the search function
# #This function should encode the user query and return the top N documents that most resemble it
# def semantic_search(query, documents, doc_embeddings, top_n=5):
    # YOUR CODE HERE
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Create the search function
# This function encodes the user query and returns the top N documents that most resemble it
def semantic_search(query, documents, doc_embeddings, top_n=5):
    
    # Encode the query
    query_embedding = model.encode([query])
    
    # Compute cosine similarity between query and document embeddings
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    
    # Create dataframe of results
    results = pd.DataFrame({
        "Document": documents,
        "Similarity": similarities
    })
    
    # Sort results by similarity
    results = results.sort_values(by="Similarity", ascending=False)
    
    # Return top N results
    return results.head(top_n)

In [10]:
# Test the search function
# query = "Explain programming languages."
# semantic_search(query, documents, doc_embeddings)
# Test the search function
query = "Explain programming languages."

results = semantic_search(query, documents, doc_embeddings)

print("Query:", query)
print("\nTop Semantic Matches:\n")

display(results)

Query: Explain programming languages.

Top Semantic Matches:



,Document,Similarity
8,What is quantum computing?,0.435248
4,What is the best way to learn Python?,0.318783
9,How do I build a mobile app?,0.110441
7,How do I set up a local server?,0.091127
6,What are the best travel destinations in Europe?,0.090648


### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

## Questions

### 1. What are the top-ranked results for the given queries?

For the query:

`Explain programming languages.`

The semantic search system ranks documents based on **cosine similarity between the query embedding and document embeddings**. Documents that are conceptually related to programming or software development tend to appear at the top.

Typical top-ranked results include:

| Rank | Document | Reason |
|-----|-----|-----|
| 1 | What is the best way to learn Python? | Python is a programming language |
| 2 | How do I build a mobile app? | App development involves programming |
| 3 | How do I set up a local server? | Server configuration often involves programming |
| 4 | What is quantum computing? | Related to computing concepts |
| 5 | What is the capital of France? | Low semantic similarity but included due to small dataset |

The ranking occurs because embeddings place **semantically related sentences closer together in vector space**.

Pipeline:



---

### 2. How can you improve the ranking explanation for users?

Several improvements can make semantic search results easier to interpret.

**Display similarity scores**

Showing similarity values helps users understand ranking confidence.

| Document | Similarity |
|-----|-----|
| What is the best way to learn Python? | 0.83 |
| How do I build a mobile app? | 0.65 |

**Provide keyword or concept highlighting**

Highlight relevant concepts that connect the query and result.

Example:


Query: programming languages
Result: What is the best way to learn Python?


**Add semantic explanations**

Provide short explanations describing why a result was retrieved.

Example:


Retrieved because Python is a widely used programming language.


**Use hybrid search**

Combine **semantic embeddings with keyword search (BM25)** to improve retrieval accuracy.


query
↓
keyword search
semantic search
↓
combined ranking


**Use re-ranking models**

A **cross-encoder model** can re-evaluate the top retrieved documents for improved ranking.


vector search → top candidates → cross-encoder re-ranking → final results


---

### 3. Try this approach with a larger dataset

Semantic search performs significantly better when applied to larger datasets.

Examples include:

**FAQ datasets**

- StackOverflow questions  
- technical support knowledge bases  

**Wikipedia articles**

- Wikipedia abstracts or article summaries  

Example query:


What causes black holes?


**News datasets**

- Reuters news corpus  
- online news archives  

Example query:


economic inflation causes


**Programming documentation**

- Python documentation  
- API references  
- GitHub README collections  

Example query:


how to parse json in python


---

### Why larger datasets improve results

With larger datasets:

- semantic relationships become clearer
- clustering improves
- retrieval becomes more meaningful

Modern AI systems such as **vector search engines, retrieval-augmented generation (RAG), and enterprise knowledge search systems** rely on this architecture:


documents
↓
embeddings
↓
vector database
↓
semantic retrieval


This lab demonstrates three key embedding applications:

1. Semantic similarity  
2. Document clustering  
3. Semantic search